In [2]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("../data/processed/cleaned_complaints.csv")

# Keep only the required columns
df = df[["clean_text"]].dropna()

# Urgency keywords list
urgent_keywords = [
    "urgent", "immediately", "asap", "as soon as possible",
    "emergency", "important", "priority", "right away",
    "resolve fast", "resolve immediately"
]

# Function to detect urgency
def detect_urgency(text):
    text = text.lower()
    for word in urgent_keywords:
        if word in text:
            return 1
    return 0

# Create urgency label
df["urgency_label"] = df["clean_text"].apply(detect_urgency)

# Check distribution
print(df["urgency_label"].value_counts())
df.head()


urgency_label
0    3826
1     534
Name: count, dtype: int64


,clean_text,urgency_label
0,summer xx xx denied mortgage loan due charge x...,0
1,many mistakes appear report without understanding,0
2,many mistakes appear report without understanding,0
3,many mistakes appear report without understanding,0
4,many mistakes appear report without understanding,0


In [3]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["urgency_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (3488,)
Test size: (872,)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words="english"
)

# Fit on train, transform both
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Train shape:", X_train_tfidf.shape)
print("TF-IDF Test shape:", X_test_tfidf.shape)


TF-IDF Train shape: (3488, 3000)
TF-IDF Test shape: (872, 3000)


In [5]:
from sklearn.linear_model import LogisticRegression

# Initialize model
urgency_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

# Train model
urgency_model.fit(X_train_tfidf, y_train)

print("Urgency classification model trained successfully!")


Urgency classification model trained successfully!


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

# Predict on test data
y_pred = urgency_model.predict(X_test_tfidf)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[728  37]
 [ 26  81]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       765
           1       0.69      0.76      0.72       107

    accuracy                           0.93       872
   macro avg       0.83      0.85      0.84       872
weighted avg       0.93      0.93      0.93       872



In [7]:
# Final urgency score (ML + Keyword)

# ML probability
ml_prob = urgency_model.predict_proba(X_test_tfidf)[:, 1]

# Keyword score (binary -> weighted)
keyword_score = y_test.values * 0.4

# Final urgency score (0–100)
final_urgency_score = (ml_prob * 0.6 + keyword_score) * 100

# Show sample
final_df = pd.DataFrame({
    "text": X_test.values[:10],
    "actual_urgency": y_test.values[:10],
    "ml_probability": ml_prob[:10],
    "final_urgency_score": final_urgency_score[:10]
})

final_df


,text,actual_urgency,ml_probability,final_urgency_score
0,made many attempts contact company already inf...,0,0.117777,7.066637
1,disputed account ideal collection service open...,0,0.226242,13.574515
2,offer xxxx points upgraded new credit card cre...,0,0.116035,6.962115
3,called see whats going fact closed account mak...,0,0.177009,10.620564
4,sent letters letter bureaus experian xxxx xxxx...,0,0.108450,6.506977
5,xx xx xxxx request investigation credit inquir...,0,0.471527,28.291604
6,explained debt collector matter already resolv...,0,0.257701,15.462051
7,send letters attached requested documents iden...,0,0.212911,12.774641
8,two different occasions husband visited doctor...,1,0.592072,75.524296
9,second written request removal block informati...,0,0.107987,6.479216
